### SQL Data Analysis

This notebook demonstrates how to query, manipulate, and analyze data using SQL. The goal is to extract meaningful insights from a relational dataset.

### Objectives 

- Connect to the database
- Query tables and inspect data
- Filter and aggregate records
- Join tables for combined analysis
- Clean and prepare data for reporting


###    

In [13]:
import pandas as pd 
import sqlite3

df = pd.read_csv('heart_cleaned.csv')
conn = sqlite3.connect('heart_data.db')
df.to_sql('patients',  conn, if_exists='replace', index=False)

302

#### Sqlte3 

It comes with python so there is nothing to install.This creates a database with one table called patients.

Put the Sql in qoutes and run it through pandas

In [14]:
#Running a query 

pd.read_sql("SELECT * FROM patients LIMIT 5",conn)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target,sex_label,target_label,has_disease,exang_label
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1,Male,No Heart Disease,0,No
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1,Male,No Heart Disease,0,No
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1,Female,No Heart Disease,0,No
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1,Male,No Heart Disease,0,No
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1,Female,No Heart Disease,0,Yes


In [15]:
#count number of records 
pd.read_sql("SELECT COUNT(*) FROM patients", conn)

,COUNT(*)
0,302


In [16]:
#count the patients in each sex goup.GROUPBY  is the SQL version of value_counts.
pd.read_sql("SELECT sex_label, COUNT(*) FROM patients GROUP BY sex_label",conn)

,sex_label,COUNT(*)
0,Female,96
1,Male,206


In [17]:
#Avarage cholesterol for patients with and without disease.
pd.read_sql("SELECT target_label, AVG(chol) FROM patients GROUP BY target_label",conn)

,target_label,AVG(chol)
0,Heart Disease,251.086957
1,No Heart Disease,242.640244


#### Average Cholesterol 

FROM FIRST PREVIEW

About 243 for "heart disease" group and about 251 for "no heart disease".Its a surprise for " heart disease" was expected to have a higher cholesterol average.

Two lessons from it:
- One variable rarely tells the whole story:Cholesterol alone is a weak signal here, which is why models combine many features.
- Always sanity-check your labels.
Some kaggle copies of the dataset are reported to have the *target* coding reversed compared with the original.

Healthy people usually reach a *higher* maximum heart rate during exercise, and chest pain during exercise(exang) is more common in people with disease.
 

In [18]:
pd.read_sql("SELECT target_label, AVG(thalach), AVG(exang) FROM patients GROUP BY target_label",conn)

,target_label,AVG(thalach),AVG(exang)
0,Heart Disease,139.101449,0.550725
1,No Heart Disease,158.378049,0.140244


In [19]:
#one more check
pd.read_sql("SELECT target_label, AVG(oldpeak), AVG(ca) FROM patients GROUP BY target_label",conn)

,target_label,AVG(oldpeak),AVG(ca)
0,Heart Disease,1.585507,1.166667
1,No Heart Disease,0.586585,0.341463


#### One More Check

From the above results.Heart disease has lower values.So target variable is probablt flipped. In the original dataset, 1 meant disease.In many kaggle copies its the other way round(1=less chance of disease)

#### CORRECTION 

*Problem*

The first results (cholesterol 243 for "heart disease" vs "No heart disease") went against what i expected.

*Check*

Thalach and exang pointed the same way, which suggested "target" is coded the opposite way in this file.

*Fix*

Swapped the labels in '02_clean.ipynb' and added a 'has_disease' column where 1 = disease.

*Result*

After rebuilding the database, the cholesterol averages are now: 251 (heart disease) and 243 (no heart disease)


In [20]:
#patients over 60 
pd.read_sql("SELECT COUNT(*) FROM patients WHERE age > 60",conn)

,COUNT(*)
0,79


In [21]:
#patients with disease in each sex group
pd.read_sql("SELECT sex_label, COUNT(*) FROM patients WHERE target_label ='Heart Disease' GROUP BY sex_label",conn)

,sex_label,COUNT(*)
0,Female,24
1,Male,114


There are less females and more males in this sample.